# Generated Knowledge Prompting for Depression Detection

**Course:** Natural Language Processing · Unit 4 — Prompt Engineering  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 12 — Prompting and In-Context Learning  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

1. Understand the *Generated Knowledge Prompting* technique (Liu et al., 2022)
2. Use an LLM to generate domain-specific exemplars for a downstream classification task
3. Compute knowledge-weighted scores over generated exemplar sets
4. Observe how injected knowledge improves classification accuracy over a naive baseline


---

## 1. Background

**Generated Knowledge Prompting** is a two-stage approach:
1. **Stage 1 — Knowledge Generation:** ask an LLM to produce relevant background  
   knowledge (exemplar sentences) for the target task.
2. **Stage 2 — Knowledge Application:** feed the generated knowledge as additional  
   context to a downstream classifier or scoring function.

In this notebook we apply the technique to **depression symptom classification**:  
ChatGPT o3-mini generates expressive sentences describing depression symptoms,  
and those sentences serve as exemplars for a score-based classifier.


---

## 2. Environment Setup

```bash
pip install openai
```


In [ ]:
# pip install openai  # run once in terminal
import os

import openai

# Load API key from environment — never hard-code secrets in a notebook
openai.api_key = os.environ.get("OPENAI_API_KEY", "your-openai-api-key-here")


---

## 3. Stage 1 — Generate Knowledge Exemplars

The prompt below instructs GPT-4o to generate diverse sentences that reflect  
depression symptoms. These sentences will serve as the 'generated knowledge'.


**Generated prompt (run externally via API):**

> You are a mental health expert helping build a symptom-aware AI model for depression detection.  
> Your task is to generate expressive and diverse example sentences (exemplars) in **English**  
> that reflect different depression symptoms (e.g. sadness, hopelessness, loss of interest, fatigue).  
> Generate 5 exemplars per symptom category, labelled clearly.  
> Output format: `[CATEGORY] sentence`


> **Note:** The exemplar generation was run via the OpenAI web interface (ChatGPT o3-mini  
> fed by ChatGPT 4o). The resulting exemplar dictionary is hard-coded in Section 4 below  
> so this notebook runs without an active API connection.


---

## 4. Stage 2 — Apply Generated Knowledge

The exemplars are stored in a Python dictionary keyed by symptom category ID.


In [ ]:
# Exemplars produced by ChatGPT o3-mini (Stage 1 output)
exemplars_dict = {
    1: [
        "I just feel so down all the time, like a heavy cloud is constantly hanging over me.",
        "Every little thing seems to make me sad these days, even the smallest setbacks.",
        "I used to be happy, but now I struggle to feel anything positive.",
        "There's a constant emptiness inside me that I can't shake off.",
        "Even on sunny days, everything feels grey and meaningless to me.",
    ],
    2: [
        "I can't see any point in trying anymore; nothing ever gets better.",
        "The future looks completely dark and I can't imagine things improving.",
        "No matter what I do, I feel like I'll never be truly happy.",
        "I've given up hoping for anything good to happen in my life.",
        "It feels like I'm stuck in a hole with no way out.",
    ],
}

for category_id, sentences in exemplars_dict.items():
    print(f"Category {category_id}: {len(sentences)} exemplars")
    for s in sentences[:2]:
        print(f"  - {s}")
    print()


> **Output interpretation:** The exemplars cover two symptom categories: persistent sadness (1) and hopelessness (2). Additional categories (loss of interest, fatigue, sleep disturbance) can be added by extending the dictionary. Diverse and expressive exemplars improve the quality of the downstream score because they cover a wider range of linguistic expressions for the same symptom.


---

## 5. Knowledge-Weighted Scoring

Given a new sentence, we compute a simple overlap-based similarity score  
against each exemplar category and assign the label with the highest score.


In [ ]:
def compute_knowledge_score(sentence: str, exemplars: list[str]) -> float:
    """Compute word-overlap similarity between sentence and a list of exemplars."""
    words = set(sentence.lower().split())
    scores = []
    for exemplar in exemplars:
        exemplar_words = set(exemplar.lower().split())
        overlap = len(words & exemplar_words)
        scores.append(overlap / max(len(words | exemplar_words), 1))
    return sum(scores) / len(scores) if scores else 0.0


def classify_with_generated_knowledge(sentence: str, exemplars_dict: dict) -> int:
    """Return the category ID with the highest knowledge score."""
    best_category = max(
        exemplars_dict,
        key=lambda cat_id: compute_knowledge_score(sentence, exemplars_dict[cat_id]),
    )
    return best_category


test_sentences = [
    "I feel hopeless and cannot see a better future.",
    "Everything feels grey and sad, I have no motivation.",
]

for sent in test_sentences:
    cat = classify_with_generated_knowledge(sent, exemplars_dict)
    print(f"Sentence: {sent}")
    print(f"  → Category: {cat}")
    print()


> **Output interpretation:** The word-overlap scorer is a baseline; in production you would use cosine similarity over sentence embeddings (e.g. from Sentence-BERT). Even this simple approach shows that generated exemplars provide useful grounding: sentences about hopelessness score higher against Category 2 exemplars than against Category 1.


---

## Summary

| Stage | Technique | Model used |
|---|---|---|
| Knowledge Generation | Prompt GPT to produce domain exemplars | GPT-4o / ChatGPT o3-mini |
| Knowledge Application | Overlap-based scoring over exemplars | Baseline (extend with embeddings) |

Generated Knowledge Prompting is particularly effective when the target domain is  
specialised (e.g. medical, legal) and labelled training data is scarce.
